In [0]:
%sql
DROP TABLE IF EXISTS bronze_fantasy_books;
DROP TABLE IF EXISTS silver_fantasy_books;

In [0]:
import requests, time
from pyspark.sql import functions as F

url = "https://openlibrary.org/search.json"

all_books = []

# Queries for fantasy book entries

# Limit each query to 500 entries, get 2500 entries total
# Note: Queries will be in default `relevance` order meaning the resulting raw bronze data table is arbitrarily selected 2500 books
limit = 500
for page in range(1, 6):
    print(f"Query from page#{page} - books {1+(page-1)*limit} to {limit+(page-1)*limit}")

    params = {
        "q": "subject:fantasy",
        "fields": "title,author_name,first_publish_year,subject,key",
        "limit": limit,
        "page": page,
    }

    headers = {"User-Agent": "book-data (annonymous)"}

    response = requests.get(url, params=params, headers=headers, timeout=30)
    response.raise_for_status()

    page_data = response.json()
    all_books.extend(page_data["docs"])

    # Open Library allows 1 query per second
    time.sleep(1)

books_df = spark.createDataFrame(all_books)
(
    books_df.withColumn("ingested_at", F.current_timestamp())
    .write.format("delta")
    .mode("overwrite")
    .saveAsTable("bronze_fantasy_books")
)

In [0]:
%sql
-- View Bronze table
SELECT
  *
FROM
  bronze_fantasy_books;

In [0]:
%sql
-- Create Silver table by casting columns to correct data types
CREATE OR REPLACE TABLE silver_fantasy_books AS
SELECT DISTINCT
  key,
  title,
  author_name,
  CAST(first_publish_year AS INT) AS first_publish_year,
  subject
FROM
  bronze_fantasy_books;

-- View Silver table
SELECT
  *
FROM
  silver_fantasy_books;

In [0]:
%sql
-- Create aggregated Gold table by Year
CREATE OR REPLACE TABLE gold_fantasy_books_by_year AS
SELECT
  first_publish_year,
  COUNT(*) AS book_count
FROM
  silver_fantasy_books
GROUP BY
  first_publish_year
ORDER BY
  first_publish_year;

-- View Gold table by year
SELECT
  *
FROM
  gold_fantasy_books_by_year;

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- Create aggregated Gold table by accredited author
-- Must unnest the array of author names first
CREATE OR REPLACE TABLE gold_fantasy_books_by_author AS
WITH unnested as (
  SELECT
    author_credited
  FROM
    silver_fantasy_books
    LATERAL VIEW EXPLODE(author_name) as author_credited
)
SELECT
  author_credited,
  COUNT(*) AS author_credit_count
FROM
  unnested
GROUP BY
  author_credited
ORDER BY
  author_credit_count desc;

-- View gold table by author
SELECT
  *
FROM
  gold_fantasy_books_by_author;

Databricks visualization. Run in Databricks to view.